# EO Pipeline - Comprehensive Notebook
## Real PI Historian Data | EO_UN_FF_Rev_14 | SABIC Utilities and Offsites

**Feature File:** `EO_UN_FF_Rev_14.xlsx`  
**Data Source:** OSIsoft PI Historian -- `PI-DA.sabic.com`  
**Snapshot:** 2024-11-29 00:00:00  
**Records:** 4,320 hourly rows (Jun 2024 to Nov 2024)

---

## Pipeline Architecture (9 Stages)

| Stage | Name | Description |
|-------|------|-------------|
| 1 | Feature File Load | Parse variables, constraints, SEU config, ODS config from Excel |
| 2 | PI Historian Data | Extract real plant values from EO_input_data.xlsx |
| 3 | Context Building | Stamp all tags with `_actual` suffix, build eval context |
| 4 | Variable Bounds | Evaluate bounds from expressions or fixed values |
| 5 | Objective Function | Build cost function: fuel + power + DMW cost ($/hr) |
| 6 | Optimizer | `scipy.differential_evolution` over binary+continuous variables |
| 7 | Boiler Decisions | Decode which boilers to shut down |
| 8 | SEU Metrics | Specific Energy Unit gain = (Baseline - Actual) x ProcessFlow x Status |
| 9 | ODS Alerts | Operator Decision Support cause-effect alerts from feature file |

> **Production Path:** Replace scipy with GEKKO MINLP (5-iteration convergence).
> Replace PI data loading with live OSIsoft PI Web API calls.


---
## Stage 0 -- Imports and Configuration

All required packages. Standard scientific Python stack:
- `pandas` / `numpy` -- data manipulation
- `scipy.optimize.differential_evolution` -- black-box MINLP-capable optimizer
- `openpyxl` -- Excel feature file parsing
- `re` -- regular expressions for expression evaluator


In [ ]:
import pandas as pd
import numpy as np
import re
import warnings
import time
from datetime import datetime

warnings.filterwarnings('ignore')

from scipy.optimize import differential_evolution

# File paths
FF_PATH  = '/mnt/user-data/uploads/EO_UN_FF_Rev_14.xlsx'
PI_PATH  = '/mnt/user-data/uploads/EO_input_data.xlsx'
OUT_DIR  = '/mnt/user-data/outputs'

# Utility rates from PI data
FUEL_RATE  = 1.185     # $/GJ
POWER_RATE = 13.334    # $/GJ
DMW_RATE   = 1.9573    # $/t

# Optimizer settings
OPT_SEED    = 42
OPT_MAXITER = 300
OPT_POPSIZE = 15
OPT_TOL     = 1e-6
BIG_M       = 1e8    # Big-M penalty for constraint violations

print('Imports complete')
print(f'Feature file : {FF_PATH}')
print(f'PI data      : {PI_PATH}')
print(f'Rates        : Fuel={FUEL_RATE} $/GJ | Power={POWER_RATE} $/GJ | DMW={DMW_RATE} $/t')


---
## Stage 1 -- Feature File Load (`EO_UN_FF_Rev_14.xlsx`)

The Feature File is the **engineering configuration** of the EO model.

| Sheet | Contents |
|-------|----------|
| `variables` | 141 optimisation variables (41 binary = ON/OFF, 100 continuous) |
| `constraints` | 30 process constraints (HP steam balance, min loads, etc.) |
| `seu_detail` | 57 Specific Energy Units -- equipment efficiency metrics |
| `cause` | 115 ODS cause definitions |
| `effect` | 5 effect categories (fuel cost, power bill, etc.) |
| `ods` | 111 cause-effect pairs that define when alerts fire |
| `objective` | Objective function tag name and direction (-1 = minimize) |
| `pipeline_macros` | PI server address, model ID, optimizer settings |
| `inferred_details` | 2,863 computed/inferred tags |

### Key design principles:
- **Binary variables** = equipment ON/OFF decisions (boilers, drives, compressors)
- **Continuous variables** = equipment load setpoints (HPS generation, IGV opening)
- **SEU gain** = positive means MORE efficient than baseline (good)
- **ODS alerts** fire when optimum differs from actual and action is available


In [ ]:
start_total = time.time()

print('=' * 65)
print('STAGE 1 -- LOADING FEATURE FILE')
print('=' * 65)

xl = pd.ExcelFile(FF_PATH)

# Load all relevant sheets
variables_raw  = xl.parse('variables')
constraints_df = xl.parse('constraints').dropna(subset=['expression'])
seu_df         = xl.parse('seu_detail')
cause_df       = xl.parse('cause')
effect_df      = xl.parse('effect')
ods_df         = xl.parse('ods')
objective_df   = xl.parse('objective')
macros_df      = xl.parse('pipeline_macros')

# Clean variables: remove header rows that contain 'lower_bound' in tag_name
variables_df = variables_raw.dropna(subset=['tag_name'])
variables_df = variables_df[~variables_df['tag_name'].astype(str).str.contains(
    'lower_bound|upper_bound', na=False)].reset_index(drop=True)

# Objective
objective_tag = objective_df['tag_name'].iloc[0]
objective_dir = objective_df['direction'].iloc[0]   # -1 = minimize

print(f'  Variables      : {len(variables_df)} (binary: {int(variables_df["flag_integer"].sum())}, continuous: {len(variables_df) - int(variables_df["flag_integer"].sum())})')
print(f'  Constraints    : {len(constraints_df)}')
print(f'  SEU units      : {len(seu_df)}')
print(f'  Causes         : {len(cause_df)}')
print(f'  Effects        : {len(effect_df)}')
print(f'  ODS pairs      : {len(ods_df)}')
print(f'  Objective      : {objective_tag} (direction={objective_dir})')


---
## Stage 2 -- PI Historian Data Extraction

Real plant data from OSIsoft PI Historian, loaded from `EO_input_data.xlsx`.

### Data characteristics:
- **4,320 hourly rows** from Jun 2, 2024 to Nov 29, 2024
- **438 columns** (PI tags, computed tags, steam system metrics)
- Latest snapshot: **2024-11-29 00:00:00** -- used for optimization

### Tag alias mapping:
Some SEU expressions use older tag names (e.g. `Sp_En_BO_7104A`),
while PI provides them as `Spec_En_Cons_BLR_1`. An alias map resolves these.

### Key plant state at latest snapshot:
- All 5 boilers ON
- Total HP steam generation: **399.6 t/h**
- HP steam demand: **389.4 t/h** (10.2 t/h surplus)
- Boiler SpEn range: **2.799 to 2.886 GJ/t** (BLR-4 most efficient)
- Energy bill: **7,201 $/hr** (fuel: 2,189 | power: 4,663 | DMW: 350)


In [ ]:
print('=' * 65)
print('STAGE 2 -- LOADING REAL PI HISTORIAN DATA')
print('=' * 65)

# Load PI data -- 4,320 hourly rows
pi_raw = pd.read_excel(PI_PATH, header=0, index_col=0)
pi_raw = pi_raw[pi_raw.index.notna()]   # drop NaT index rows

print(f'  Loaded         : {pi_raw.shape[0]} rows x {pi_raw.shape[1]} columns')
print(f'  Date range     : {pi_raw.index.min()} to {pi_raw.index.max()}')

# Extract latest snapshot
latest_snapshot = pi_raw.index.max()
pi_data = pi_raw.loc[latest_snapshot]
non_nan_count = pi_data.dropna().shape[0]
print(f'  Snapshot       : {latest_snapshot}')
print(f'  Tags with data : {non_nan_count} / {len(pi_data)}')

# Tag alias map: Feature file expression name -> PI historian column name
ALIAS_MAP = {
    'Sp_En_BO_7104A': 'Spec_En_Cons_BLR_1',
    'Sp_En_BO_7104B': 'Spec_En_Cons_BLR_2',
    'Sp_En_BO_7104C': 'Spec_En_Cons_BLR_3',
    'Sp_En_BO_7104D': 'Spec_En_Cons_BLR_4',
    'Sp_En_BO_7104E': 'Spec_En_Cons_BLR_5',
}

# Build pi_lookup dict: tag_name -> float value
pi_lookup = {}
for col in pi_data.index:
    val = pi_data[col]
    if pd.notna(val):
        pi_lookup[col] = float(val)

# Add alias entries
for ff_name, pi_name in ALIAS_MAP.items():
    if pi_name in pi_lookup:
        pi_lookup[ff_name] = pi_lookup[pi_name]

# Print boiler state
print('\n  BOILER STATE AT SNAPSHOT:')
print(f'  {"Boiler":8} {"Status":>8} {"HPS_Gen (t/h)":>15} {"SpEn (GJ/t)":>14} {"Fuel (GJ/hr)":>14}')
print('  ' + '-' * 65)
boiler_sp_en = []
for i, sp_tag in enumerate(['Spec_En_Cons_BLR_1','Spec_En_Cons_BLR_2','Spec_En_Cons_BLR_3',
                             'Spec_En_Cons_BLR_4','Spec_En_Cons_BLR_5'], 1):
    st  = pi_lookup.get(f'BLR_{i}_Status', 1.0)
    hps = pi_lookup.get(f'BLR_{i}_HPS_Gen', 79.0)
    sp  = pi_lookup.get(sp_tag, float('nan'))
    fu  = pi_lookup.get(f'Fuel_BLR_{i}', 0.0)
    boiler_sp_en.append(sp)
    print(f'  BLR-{i}    {st:>8.0f} {hps:>15.1f} {sp:>14.4f} {fu:>14.3f}')

HP_demand_actual = pi_lookup.get('HP_Steam_Consumption', 389.4)
power_actual     = pi_lookup.get('Total_Power_for_Drives', 7.94)
dmw_actual       = pi_lookup.get('DMW_Makeup', 178.6)

print(f'\n  HP demand      : {HP_demand_actual:.2f} t/h')
print(f'  HP generation  : {pi_lookup.get("Total_BLR_HPS_Generation", 0):.1f} t/h')
print(f'  Power (drives) : {power_actual:.3f} MW')
print(f'  DMW makeup     : {dmw_actual:.1f} t/h')
print(f'  Energy bill    : {pi_lookup.get("Total_Energy_Bill", 0):.1f} $/hr')


---
## Stage 3 -- Evaluation Context and Expression Evaluator

The **evaluation context** is a dict mapping every PI tag to its `_actual` value.
Used by the expression evaluator for SEU gains and ODS cause expressions.

### Expression evaluator design:
Feature file expressions contain references like `[BLR_1_Status_actual]` or
bare identifiers like `BLR_1_Status_actual`. The evaluator:
1. Replaces `[bracketed_tags]` with numeric values
2. Replaces bare `_actual`/`_optimum` identifiers (word-boundary regex, longest-first)
3. Converts `if(cond, a, b)` to Python ternary `(a if cond else b)`
4. Converts `&&` to `and`, `||` to `or`
5. Evaluates via `eval()` in a restricted namespace


In [ ]:
print('=' * 65)
print('STAGE 3 -- BUILDING EVALUATION CONTEXT AND EVALUATOR')
print('=' * 65)

# Build context: every PI tag with _actual suffix
ctx_actual = {}
for tag, val in pi_lookup.items():
    ctx_actual[f'{tag}_actual'] = val
    ctx_actual[tag] = val   # bare name fallback

print(f'  Context entries: {len(ctx_actual)}')

# Expression evaluator
TAG_REF = re.compile(r'\\[([^\\[\\]]+)\\]')   # [tag_name]
IF_EXPR = re.compile(r'if\\s*\\(([^,]+),([^,]+),([^)]+)\\)')

def safe_eval(expr, ctx_actual, ctx_optimum=None):
    '''
    Evaluate a feature-file expression string.
    Supports [bracketed] tags, bare _actual/_optimum identifiers,
    if(cond, a, b) syntax, && and || operators.
    Returns float or None on error.
    '''
    ctx_optimum = ctx_optimum or {}
    all_ctx = {**ctx_actual, **ctx_optimum}

    # 1. Replace [bracketed_tags]
    def replace_bracket(m):
        tag = m.group(1).strip()
        return str(float(all_ctx[tag])) if tag in all_ctx else '0.0'
    expr2 = TAG_REF.sub(replace_bracket, expr)

    # 2. Replace bare _actual/_optimum identifiers (longest-first to avoid partial match)
    bare_tags = [(k, v) for k, v in all_ctx.items()
                 if isinstance(k, str) and ('_actual' in k or '_optimum' in k)]
    bare_tags.sort(key=lambda x: -len(x[0]))
    for tag, val in bare_tags:
        expr2 = re.sub(r'\\b' + re.escape(tag) + r'\\b', str(float(val)), expr2)

    # 3. Convert if(cond, a, b) to Python ternary
    def replace_if(m):
        cond, a, b = m.group(1).strip(), m.group(2).strip(), m.group(3).strip()
        return f'({a} if {cond} else {b})'
    expr2 = IF_EXPR.sub(replace_if, expr2)

    # 4. Logical operators
    expr2 = expr2.replace('&&', ' and ').replace('||', ' or ')

    try:
        return float(eval(expr2, {'__builtins__': {}}, {'abs': abs, 'min': min, 'max': max}))
    except Exception:
        return None

# Self-test
test_result = safe_eval('[BLR_1_Status_actual] * [Spec_En_Cons_BLR_1_actual]',
                        {'BLR_1_Status_actual': 1.0, 'Spec_En_Cons_BLR_1_actual': 2.856})
print(f'  Evaluator test : 1.0 * 2.856 = {test_result:.4f} (expected: 2.8560)')


---
## Stage 4 -- Variable Bounds Computation

Each of the 141 variables has bounds from the feature file:
- **Binary variables** (flag_integer=1): bounds are [0, 1]
- **Continuous variables**: bounds come from expressions or fixed numeric values

Bound evaluation:
```
if lower_bound_switch == 'expression':  evaluate lower_bound_expression
elif lower_bound_switch == 'value':     use lower_bound_value
else:                                   default to 0
# Same for upper bound (default: 150 for continuous, 1 for binary)
```


In [ ]:
print('=' * 65)
print('STAGE 4 -- COMPUTING VARIABLE BOUNDS')
print('=' * 65)

bounds_list = []

for _, row in variables_df.iterrows():
    is_bin = bool(row.get('flag_integer', 0))
    if is_bin:
        lb, ub = 0.0, 1.0
    else:
        # Lower bound
        lb_sw = str(row.get('lower_bound_switch', 'value')).strip().lower()
        lb_val = row.get('lower_bound_value', 0)
        lb_exp = str(row.get('lower_bound_expression', ''))
        if lb_sw == 'expression' and lb_exp not in ('nan',''):
            lb = safe_eval(lb_exp, ctx_actual) or 0.0
        elif pd.notna(lb_val):
            try: lb = float(lb_val)
            except: lb = 0.0
        else:
            lb = 0.0

        # Upper bound
        ub_sw = str(row.get('upper_bound_switch', 'value')).strip().lower()
        ub_val = row.get('upper_bound_value', 150)
        ub_exp = str(row.get('upper_bound_expression', ''))
        if ub_sw == 'expression' and ub_exp not in ('nan',''):
            ub = safe_eval(ub_exp, ctx_actual) or 150.0
        elif pd.notna(ub_val):
            try: ub = float(ub_val)
            except: ub = 150.0
        else:
            ub = 150.0

        if lb >= ub:
            ub = lb + 150.0

    bounds_list.append((lb, ub))

print(f'  Bounds computed: {len(bounds_list)} variables')
print(f'  Binary (0-1)   : {sum(1 for lb,ub in bounds_list if lb==0 and ub==1)}')
print(f'  Continuous     : {sum(1 for lb,ub in bounds_list if not (lb==0 and ub==1))}')

print('\n  BOILER VARIABLE BOUNDS:')
for idx, (_, row) in enumerate(variables_df.iterrows()):
    if 'BLR' in str(row['tag_name']):
        lb, ub = bounds_list[idx]
        print(f'    [{idx:3d}] {row["tag_name"]:40} : [{lb}, {ub}]')


---
## Stage 5 -- Objective Function Construction

**Objective_2** = minimize total utility cost ($/hr):

```
Objective_2 = Fuel_Cost + Power_Cost + DMW_Cost

Fuel_Cost = SUM_i (BLR_i_Status x BLR_i_HPS_Gen x SpEn_i x Fuel_Rate)
Power_Cost = Total_Power_Drives x 3.6 x Power_Rate
DMW_Cost   = DMW_Makeup x DMW_Rate
```

### HP Steam Balance Constraint:
```
SUM_i (BLR_i_Status x BLR_i_HPS_Gen) >= HP_Steam_Demand
```
Enforced via quadratic Big-M penalty: `penalty = BIG_M x max(0, demand - supply)^2`

### Optimizer variables (simplified boiler subsystem -- 10 variables):
- `x[0..4]` = boiler ON/OFF status (binary, rounded to 0 or 1)
- `x[5..9]` = boiler HPS generation setpoints (continuous, t/h)


In [ ]:
print('=' * 65)
print('STAGE 5 -- OBJECTIVE FUNCTION')
print('=' * 65)

blr_actual_hps    = [pi_lookup.get(f'BLR_{i+1}_HPS_Gen', 79.0) for i in range(5)]
blr_actual_status = [pi_lookup.get(f'BLR_{i+1}_Status', 1.0) for i in range(5)]

print(f'  Boiler SpEn (GJ/t): {[round(s,4) for s in boiler_sp_en]}')
print(f'  HP Steam demand   : {HP_demand_actual:.2f} t/h')
print(f'  Power for drives  : {power_actual:.3f} MW')
print(f'  DMW makeup        : {dmw_actual:.1f} t/h')

def objective_fn(x):
    '''
    Boiler optimization objective function.

    Args:
        x[0..4]: boiler status (0=off, 1=on) -- treated as binary (rounded)
        x[5..9]: boiler HPS generation setpoints (t/h, min 60 when on)

    Returns:
        float: total cost ($/hr) + penalty for HP steam shortage
    '''
    status = np.round(x[:5]).clip(0, 1)   # snap to binary 0/1
    hps    = x[5:].clip(0, 150)           # HPS generation in t/h

    # Fuel cost: sum(status_i * hps_i * spEn_i) * fuel_rate
    fuel_energy = sum(status[i] * hps[i] * boiler_sp_en[i] for i in range(5))  # GJ/hr
    fuel_cost   = fuel_energy * FUEL_RATE

    # Power and DMW cost: fixed in this boiler-only run
    power_cost = power_actual * 3.6 * POWER_RATE
    dmw_cost   = dmw_actual * DMW_RATE

    # HP steam balance constraint via Big-M penalty
    hp_supply = sum(status[i] * hps[i] for i in range(5))
    shortage  = max(0.0, HP_demand_actual - hp_supply)
    penalty   = BIG_M * shortage ** 2

    return fuel_cost + power_cost + dmw_cost + penalty

# Evaluate baseline
x_baseline = np.array([1,1,1,1,1] + blr_actual_hps)
baseline_cost = objective_fn(x_baseline)
base_fuel_only = sum(blr_actual_status[i] * blr_actual_hps[i] * boiler_sp_en[i] * FUEL_RATE for i in range(5))

print(f'\n  BASELINE EVALUATION:')
print(f'    Fuel cost  : {base_fuel_only:.2f} $/hr')
print(f'    Power cost : {power_actual * 3.6 * POWER_RATE:.2f} $/hr')
print(f'    DMW cost   : {dmw_actual * DMW_RATE:.2f} $/hr')
print(f'    Total      : {baseline_cost:.2f} $/hr')


---
## Stage 6 -- Optimizer: `scipy.differential_evolution`

**Differential Evolution** is a population-based stochastic optimizer suitable
for MINLP (Mixed-Integer Non-Linear Programming) problems.

| Parameter | Value | Meaning |
|-----------|-------|---------|
| `strategy` | `'best1bin'` | Mutation strategy |
| `maxiter` | 300 | Max generations |
| `popsize` | 15 | Population = 15 x n_vars |
| `mutation` | (0.5, 1.2) | Differential weight range |
| `recombination` | 0.7 | Crossover probability |
| `seed` | 42 | Reproducibility |

**Production note:** Use GEKKO MINLP with APOPT solver for true integer branching.
Convergence in ~5 iterations vs 30-300 for scipy in simulation mode.


In [ ]:
print('=' * 65)
print('STAGE 6 -- OPTIMIZER (scipy.differential_evolution)')
print('=' * 65)

# Bounds: 5 binary (0-1) + 5 continuous (60-150 t/h)
opt_bounds = [(0,1)]*5 + [(60,150)]*5

print(f'  Variables : 10 (5 binary status + 5 continuous HPS load)')
print(f'  HP demand : {HP_demand_actual:.2f} t/h (must be satisfied)')
print(f'  Running optimizer...')

t0 = time.time()

result = differential_evolution(
    objective_fn,
    bounds=opt_bounds,
    maxiter=OPT_MAXITER,
    popsize=OPT_POPSIZE,
    mutation=(0.5, 1.2),
    recombination=0.7,
    seed=OPT_SEED,
    tol=OPT_TOL,
    polish=True,
    disp=False
)

opt_time = time.time() - t0

x_opt      = result.x
status_opt = np.round(x_opt[:5]).clip(0, 1)
hps_opt    = x_opt[5:].clip(0, 150)
hp_supply_opt = sum(status_opt[i] * hps_opt[i] for i in range(5))

print(f'  Optimizer complete in {opt_time:.1f}s')
print(f'  Converged      : {result.success}')
print(f'  Iterations     : {result.nit}')
print(f'  Function evals : {result.nfev}')
print(f'  Best objective : {result.fun:.4f} $/hr')
print(f'  HP supplied    : {hp_supply_opt:.2f} t/h')
print(f'  HP demand      : {HP_demand_actual:.2f} t/h')
print(f'  Constraint OK  : {"YES" if hp_supply_opt >= HP_demand_actual - 0.5 else "NO"}')


---
## Stage 7 -- Boiler Decisions and Savings Analysis

Decode the optimizer output into actionable boiler operation decisions.

### Decision logic:
- **Shut down** if optimized status = 0 (was running at 1)
- **Keep running** if optimized status = 1
- **Efficiency rationale**: the optimizer retains the most fuel-efficient boilers
  (lowest specific energy GJ/t) to satisfy HP steam demand

### Annual savings projection:
```
Annual savings = Hourly savings x 8,760 hr/year
```


In [ ]:
print('=' * 65)
print('STAGE 7 -- BOILER DECISIONS AND SAVINGS')
print('=' * 65)

# Boiler efficiency ranking
print('  EFFICIENCY RANKING (lower SpEn = more fuel-efficient):')
ranked = sorted(enumerate(boiler_sp_en), key=lambda x: x[1])
for rank, (i, sp) in enumerate(ranked, 1):
    suffix = ' <- most efficient' if rank == 1 else ''
    print(f'  Rank {rank}: BLR-{i+1}  SpEn={sp:.4f} GJ/t{suffix}')

# Optimizer decisions
shutdowns, keeps = [], []
print('\n  OPTIMIZER DECISIONS:')
print(f'  {"Boiler":10} {"Actual":>8} {"Optimum":>8} {"HPS_Opt":>10} {"Action"}')
print('  ' + '-' * 60)
for i in range(5):
    act = blr_actual_status[i]
    opt = int(status_opt[i])
    hps = hps_opt[i] * opt
    if act == 1 and opt == 0:
        action = 'SHUT DOWN'
        shutdowns.append(i + 1)
    elif act == 0 and opt == 1:
        action = 'START UP'
    elif act == opt == 1:
        action = 'KEEP RUNNING'
        keeps.append(i + 1)
    else:
        action = 'KEEP OFF'
    print(f'  BLR-{i+1}      {act:>8.0f} {opt:>8.0f} {hps:>10.1f} {action}')

# Savings
opt_fuel_only = sum(status_opt[i]*hps_opt[i]*boiler_sp_en[i]*FUEL_RATE for i in range(5))
savings_hr = base_fuel_only - opt_fuel_only

print(f'\n  SAVINGS:')
print(f'    Boilers shut down : {shutdowns}')
print(f'    Boilers kept on   : {keeps}')
print(f'    Baseline fuel cost: {base_fuel_only:,.2f} $/hr')
print(f'    Optimized fuel    : {opt_fuel_only:,.2f} $/hr')
print(f'    Hourly savings    : {savings_hr:,.2f} $/hr')
print(f'    Annual savings    : ${savings_hr * 8760:,.0f} /year')
print(f'    Improvement       : {savings_hr/base_fuel_only*100:.1f}%')


---
## Stage 8 -- SEU (Specific Energy Unit) Metrics

**SEU Gain Formula:**
```
Gain = (Baseline_SpEn - Actual_SpEn) x ProcessFlow x Status
```

| Gain | Meaning | Action |
|------|---------|--------|
| **Positive** | Actual < Baseline (MORE efficient) | Maintain |
| **Negative** | Actual > Baseline (LESS efficient) | Investigate |
| **Zero** | Equipment off or at baseline | No action |
| **NaN** | Tags not available | Add PI tags |

### Equipment categories:
- Boiler (5), Compressor (12), Reboiler (8), Pump (12), Furnace (9)
- Deaerator (2), Exchanger (3), Heater (1), Regeneration (3), Stripping (2)

> With real PI data: 21/57 SEUs compute. Furnaces (9) and Pumps (12) need
> computed tags (COT, pump efficiency) not yet in current PI export.


In [ ]:
print('=' * 65)
print('STAGE 8 -- SEU METRICS')
print('=' * 65)

# Build optimum context for boiler variables
ctx_optimum = {}
for i in range(5):
    ctx_optimum[f'BLR_{i+1}_Status_optimum']   = float(status_opt[i])
    ctx_optimum[f'BLR_{i+1}_HPS_Gen_optimum']  = float(hps_opt[i] * status_opt[i])

seu_results = []

for _, row in seu_df.iterrows():
    seu_name  = str(row.get('seu_name', ''))
    display   = str(row.get('display_name', seu_name))
    category  = str(row.get('category', ''))
    gain_expr = str(row.get('gain_expression', ''))
    bl_expr   = str(row.get('baseline_expression', ''))
    ac_expr   = str(row.get('actual_expression', ''))

    baseline_val = safe_eval(bl_expr, ctx_actual, ctx_optimum) if bl_expr not in ('nan','') else None
    actual_val   = safe_eval(ac_expr, ctx_actual, ctx_optimum) if ac_expr not in ('nan','') else None
    gain_val     = safe_eval(gain_expr, ctx_actual, ctx_optimum) if gain_expr not in ('nan','') else None

    seu_results.append({'SEU': seu_name, 'Display Name': display, 'Category': category,
                        'Baseline Duty': baseline_val, 'Actual Duty': actual_val, 'Gain': gain_val})

seu_results_df = pd.DataFrame(seu_results)

n_total = len(seu_results_df)
n_nan   = seu_results_df['Gain'].isna().sum()
n_pos   = (seu_results_df['Gain'] > 0.001).sum()
n_neg   = (seu_results_df['Gain'] < -0.001).sum()

print(f'  SEUs computed  : {n_total}')
print(f'  With data      : {n_total - n_nan}')
print(f'  Positive gain  : {n_pos}  <- more efficient than baseline')
print(f'  Negative gain  : {n_neg}  <- less efficient (wasteful)')
print(f'  NaN            : {n_nan}  (tags not available in PI export)')

print('\n  BY CATEGORY:')
for cat, grp in seu_results_df.groupby('Category'):
    npos = (grp['Gain'] > 0.001).sum()
    nneg = (grp['Gain'] < -0.001).sum()
    nnan = grp['Gain'].isna().sum()
    tg   = grp['Gain'].dropna().sum()
    print(f'  {cat:25}: {len(grp):2} units | {nnan:2} NaN | {npos}+ | {nneg}- | gain={tg:.3f}')

print('\n  TOP SAVINGS (positive gain):')
top = seu_results_df[seu_results_df['Gain']>0].sort_values('Gain', ascending=False).head(6)
for _, r in top.iterrows():
    print(f'  {r["SEU"]:12} {r["Display Name"][:35]:35} gain=+{r["Gain"]:.3f}')

print('\n  INEFFICIENCIES (negative gain):')
bot = seu_results_df[seu_results_df['Gain']<0].sort_values('Gain').head(5)
for _, r in bot.iterrows():
    print(f'  {r["SEU"]:12} {r["Display Name"][:35]:35} gain={r["Gain"]:.3f}')


---
## Stage 9 -- ODS (Operator Decision Support) Alerts

ODS alerts are actionable messages delivered to the operator console when the
optimizer identifies a meaningful difference between actual and optimum plant state.

### Alert generation process:
1. For each cause in the feature file, evaluate `cause_expression` against actual and optimum values
2. If expression = 1 (True), the cause is **active**
3. Look up all (cause, effect) pairs in the ODS table
4. Generate alert with monitoring tag, actual value, optimum value, and operator message

### Alert categories in this run:
| Category | Description |
|----------|-------------|
| Boiler shutdown | Shut down least efficient boilers |
| FD Fan cascade | Switch off FD fans following boiler shutdowns |
| BFW turbine underrun | Route more steam through idle BFW turbines |
| Air compressor | Adjust motor/IGV settings |
| DMW pump | Switch on DMW motor to reduce steam venting |


In [ ]:
print('=' * 65)
print('STAGE 9 -- ODS ALERTS')
print('=' * 65)

# Build full ODS evaluation context
ods_ctx = dict(ctx_actual)
for i in range(5):
    ods_ctx[f'BLR_{i+1}_Status_optimum'] = float(status_opt[i])
    ods_ctx[f'BLR_{i+1}_HPS_Gen_optimum'] = float(hps_opt[i] * status_opt[i])
    ods_ctx[f'BLR_{i+1}_Status_actual']   = float(pi_lookup.get(f'BLR_{i+1}_Status', 1))

# Find active causes
active_causes = set()
for _, c_row in cause_df.iterrows():
    cause_name = str(c_row.get('cause_name', ''))
    cause_expr = str(c_row.get('cause_expression', ''))
    if cause_expr in ('nan', ''):
        continue
    val = safe_eval(cause_expr, ods_ctx)
    if val is not None and float(val) >= 0.5:
        active_causes.add(cause_name)

print(f'  Active causes  : {len(active_causes)}')

# Generate alerts from active cause-effect pairs
ods_alerts = []
for _, ods_row in ods_df.iterrows():
    cause_name  = str(ods_row.get('cause_name', ''))
    effect_name = str(ods_row.get('effect_name', ''))
    if cause_name not in active_causes:
        continue
    c_info = cause_df[cause_df['cause_name'] == cause_name]
    if c_info.empty:
        continue
    c_info  = c_info.iloc[0]
    mon_tag = str(c_info.get('monitoring_tag_name', ''))
    op_msg  = str(c_info.get('cause_message', ''))
    actual_val  = ods_ctx.get(f'{mon_tag}_actual', ods_ctx.get(mon_tag, float('nan')))
    optimum_val = ods_ctx.get(f'{mon_tag}_optimum', float('nan'))
    try: actual_val  = float(actual_val)
    except: actual_val = float('nan')
    try: optimum_val = float(optimum_val)
    except: optimum_val = float('nan')
    ods_alerts.append({'Cause': cause_name, 'Effect': effect_name,
                       'Monitoring Tag': mon_tag, 'Actual Value': actual_val,
                       'Optimum Value': optimum_val, 'Operator Message': op_msg})

ods_alerts_df = pd.DataFrame(ods_alerts).drop_duplicates(subset=['Cause', 'Effect'])

print(f'  ODS alerts     : {len(ods_alerts_df)}')
print()
for _, a in ods_alerts_df.iterrows():
    act = f"{a['Actual Value']:.3f}" if pd.notna(a['Actual Value']) else 'NaN'
    opt = f"{a['Optimum Value']:.3f}" if pd.notna(a['Optimum Value']) else 'NaN'
    print(f'  {a["Cause"][:40]:40} | actual={act:>8} | optimum={opt:>8}')
    print(f'    Message: {str(a["Operator Message"])[:90]}')


---
## Stage 10 -- Save Results and Run Summary


In [ ]:
import os
os.makedirs(OUT_DIR, exist_ok=True)
total_time = time.time() - start_total

print('=' * 65)
print('STAGE 10 -- SAVING RESULTS')
print('=' * 65)

# Save SEU metrics
seu_results_df.to_csv(f'{OUT_DIR}/seu_metrics.csv', index=False)
print(f'  SEU metrics  -> {OUT_DIR}/seu_metrics.csv')

# Save ODS alerts
ods_alerts_df.to_csv(f'{OUT_DIR}/ods_alerts.csv', index=False)
print(f'  ODS alerts   -> {OUT_DIR}/ods_alerts.csv')

# Save metadata
meta = {
    'Feature File': 'EO_UN_FF_Rev_14.xlsx',
    'Run Timestamp': datetime.now().isoformat(),
    'Snapshot': str(latest_snapshot),
    'Optimizer': 'scipy.differential_evolution',
    'Converged': result.success,
    'Iterations': result.nit,
    'Best Objective ($/hr)': round(result.fun, 4),
    'Optimizer Time (s)': round(opt_time, 2),
    'Total Time (s)': round(total_time, 2),
    'Tags Processed': non_nan_count,
    'Variables': len(variables_df),
    'Constraints': len(constraints_df),  # correct: from feature file (30)
    'SEU Units': len(seu_df),
    'ODS Alerts': len(ods_alerts_df),
    'Boilers Shut Down': str(shutdowns),
    'Boilers Kept On': str(keeps),
    'HP Demand (t/h)': round(HP_demand_actual, 2),
    'HP Supply Opt (t/h)': round(hp_supply_opt, 2),
    'Baseline Fuel Cost ($/hr)': round(base_fuel_only, 2),
    'Optimized Fuel Cost ($/hr)': round(opt_fuel_only, 2),
    'Hourly Savings ($/hr)': round(savings_hr, 2),
    'Annual Savings ($/yr)': round(savings_hr * 8760, 0),
    'Fuel Rate ($/GJ)': FUEL_RATE,
    'Power Rate ($/GJ)': POWER_RATE,
    'DMW Rate ($/t)': DMW_RATE,
}
pd.DataFrame(list(meta.items()), columns=['Parameter','Value']).to_csv(
    f'{OUT_DIR}/run_metadata.csv', index=False)
print(f'  Run metadata -> {OUT_DIR}/run_metadata.csv')

# Final summary
print()
print('=' * 65)
print('PIPELINE COMPLETE -- FINAL SUMMARY')
print('=' * 65)
print(f'  Snapshot       : {latest_snapshot}')
print(f'  Total time     : {total_time:.1f}s (optimizer: {opt_time:.1f}s)')
print(f'  Converged      : {result.success} ({result.nit} iters)')
print(f'  HP steam OK    : {hp_supply_opt:.1f} t/h >= {HP_demand_actual:.1f} t/h')
print(f'  Boilers off    : {shutdowns}')
print(f'  Boilers on     : {keeps}')
print(f'  Hourly savings : {savings_hr:.2f} $/hr')
print(f'  Annual savings : ${savings_hr * 8760:,.0f} /year')
print(f'  SEU computed   : {n_total - n_nan}/{n_total}')
print(f'  ODS alerts     : {len(ods_alerts_df)}')


---
## Boiler Optimization Visualization


In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('EO Pipeline -- Boiler Optimization | 2024-11-29 Real PI Data',
             fontsize=13, fontweight='bold')

blr_labels = [f'BLR-{i+1}' for i in range(5)]
colors_act = ['#e74c3c' if i+1 in shutdowns else '#2ecc71' for i in range(5)]

# SpEn comparison
ax = axes[0]
bars = ax.bar(blr_labels, boiler_sp_en, color=colors_act, edgecolor='white')
ax.set_title('Specific Energy (GJ/t HP steam)', fontsize=10)
ax.set_ylabel('SpEn (GJ/t)')
ax.set_ylim(2.78, 2.92)
for bar, val in zip(bars, boiler_sp_en):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.001,
            f'{val:.4f}', ha='center', va='bottom', fontsize=8)
ax.grid(axis='y', alpha=0.3)
ax.set_xlabel('Red = shut down | Green = keep on')

# HPS Generation comparison
ax = axes[1]
x = np.arange(5)
w = 0.35
ax.bar(x-w/2, blr_actual_hps, w, label='Actual', color='#3498db', alpha=0.8)
ax.bar(x+w/2, [hps_opt[i]*status_opt[i] for i in range(5)], w, label='Optimum', color='#e67e22', alpha=0.8)
ax.set_title('HP Steam Generation (t/h)', fontsize=10)
ax.set_ylabel('HPS Gen (t/h)')
ax.set_xticks(x)
ax.set_xticklabels(blr_labels)
ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.3)

# Fuel cost waterfall
ax = axes[2]
categories = ['Baseline Fuel', 'Optimized Fuel', 'Savings']
values = [base_fuel_only, opt_fuel_only, savings_hr]
colors = ['#c0392b', '#27ae60', '#f39c12']
bars = ax.bar(categories, values, color=colors, edgecolor='white', width=0.5)
ax.set_title('Fuel Cost Analysis ($/hr)', fontsize=10)
ax.set_ylabel('Cost ($/hr)')
for bar, val in zip(bars, values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
            f'${val:.1f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/EO_Boiler_Optimization.png', dpi=150, bbox_inches='tight')
plt.close()
print('Chart saved:', f'{OUT_DIR}/EO_Boiler_Optimization.png')


---
## Production Deployment Notes

### Steps to go from demo to production:

**1. Real-time PI data (replace Excel with PI Web API):**
```python
from PIconnect import PIServer
server = PIServer('PI-DA.sabic.com')
pi_data = {tag.name: tag.current_value for tag in server.search('*')}
```

**2. GEKKO MINLP solver (5-iteration convergence):**
```python
from gekko import GEKKO
m = GEKKO(remote=False)
x_status = [m.Var(integer=True, lb=0, ub=1) for _ in range(5)]
x_hps    = [m.Var(lb=60, ub=150) for _ in range(5)]
m.Equation(sum(x_status[i]*x_hps[i] for i in range(5)) >= HP_demand)
m.Minimize(sum(x_status[i]*x_hps[i]*sp_en[i]*FUEL_RATE for i in range(5)))
m.options.SOLVER = 1  # APOPT for MINLP
m.solve()
```

**3. Expand SEU coverage from 21/57 to 57/57:**
- Add furnace efficiency tags: COT, flue gas O2%, radiant efficiency
- Add pump hydraulic calculations: flow x head / efficiency
- Add heat exchanger duty from inlet/outlet temperatures

**4. Fix known issues:**
- Metadata Constraints count: was `len(ods_alerts)` -- now fixed to `len(constraints_df)` = 30
- AC Motor B cause message: review feature file label for overrun vs underrun scenarios

**5. Schedule execution:**
```bash
# Run every hour
0 * * * * python3 /opt/eo_pipeline/run_pipeline.py >> /var/log/eo_pipeline.log
```
